# Dreamer con política aleatoria

¿Qué modelo del mundo aprende Dreamer si la política que junta los datos es **siempre aleatoria**? Este notebook entrena solo el world model de [r2dreamer-cookie](https://github.com/FabriRandon/r2dreamer-cookie) en la galleta, con acciones al azar durante todo el run.

Diferencias con el notebook normal (`colab.ipynb`):

- `trainer.random_policy=True`: cada acción se elige uniformemente al azar; el actor nunca actúa.
- `model.train_actor_critic=False`: el actor y el crítico no se entrenan, porque no se usan. Esto también evita que sus gradientes lleguen al world model, y hace cada actualización más rápida.
- `model.rep_loss=dreamer`: usa el decoder de DreamerV3 en vez de la pérdida de R2-Dreamer, que no reconstruye imágenes. Sin decoder no se puede *ver* lo que el modelo imagina.
- `trainer.video_pred_log=True`: guarda videos de lo que el modelo predice, comparados con lo que de verdad pasó.

Si la sesión se corta, vuelve a correr los pasos 1 a 6 y después el 9 con el mismo `LOGDIR`: el run se retoma solo desde el último checkpoint.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.

## 1. Revisar la GPU que tocó

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Montar Drive

Aquí van los checkpoints, para que sobrevivan al corte de sesión.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Clonar el repo (o actualizarlo si ya está)

In [ ]:
%%bash
if [ -d /content/r2dreamer-cookie ]; then
  cd /content/r2dreamer-cookie && git pull --ff-only
else
  git clone https://github.com/FabriRandon/r2dreamer-cookie.git /content/r2dreamer-cookie
fi

## 4. Instalar

r2dreamer necesita Python 3.11 y Colab suele traer una versión más nueva, así que se crea un entorno aparte con `uv`. Toma unos minutos la primera vez; hay que repetirlo en cada sesión nueva.

In [ ]:
%%bash
pip install -q uv
cd /content/r2dreamer-cookie
uv venv --python 3.11 .venv
uv pip install --python .venv/bin/python -e ".[cookie]" 2>&1 | tail -3

## 5. Comprobar que todo quedó bien

In [ ]:
%%bash
cd /content/r2dreamer-cookie
.venv/bin/python - <<'EOF'
import torch, gymnasium, cookie_env
from envs.cookie import Cookie
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU")
a = torch.randn(256, 256, device="cuda", dtype=torch.float16)
print("float16 ok:", bool(torch.isfinite((a @ a).sum())))
print("galleta ok:", Cookie("partial").reset()["image"].shape)
EOF

## 6. Configurar el run

- `LOGDIR` va en Drive. Para retomar un run cortado, no lo cambies.
- `PASOS`: con acciones al azar no se espera que el agente resuelva nada, solo que el modelo aprenda cómo funciona el mundo. 100k pasos debería alcanzar para ver qué aprende; se puede alargar después retomando con un número mayor.
- `TRAIN_RATIO`: 512 es lo mismo que en los runs normales de la galleta, así la comparación es justa. Con 128 corre unas 4 veces más rápido, a cambio de menos actualizaciones por dato.

`OPCIONES` son las que hacen que la política sea aleatoria; la prueba corta y el run completo usan las mismas.

In [ ]:
LOGDIR = "/content/drive/MyDrive/galleta/random_partial_01"
PASOS = 100_000
TRAIN_RATIO = 512

OPCIONES = (
    f"env=cookie env.task=cookie_partial env.train_ratio={TRAIN_RATIO} env.eval_episode_num=0 "
    "trainer.random_policy=True model.train_actor_critic=False "
    "model.rep_loss=dreamer trainer.video_pred_log=True"
)


def ultimo_video(logdir):
    """Muestra el último video de predicción guardado en logdir."""
    from IPython.display import Image, display
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

    eventos = EventAccumulator(logdir, size_guidance={"images": 0})
    eventos.Reload()
    videos = eventos.Images("open_loop") if "open_loop" in eventos.Tags()["images"] else []
    if not videos:
        print("Todavía no hay videos de predicción en", logdir)
        return
    ultimo = videos[-1]
    print(f"Paso {ultimo.step} ({len(videos)} videos guardados)")
    display(Image(data=ultimo.encoded_image_string, format="gif"))

## 7. Prueba corta (unos 10 minutos)

Antes de lanzar el run largo: corre 1500 pasos en disco local (no en Drive) y comprueba que todo funcione, cuánto tardaría el run completo y que el video de predicción se genere. Si la segunda celda muestra una estimación y un video, está todo listo para el paso 9.

In [ ]:
!rm -rf /content/prueba_random
!cd /content/r2dreamer-cookie && timeout 1200 .venv/bin/python -u train.py {OPCIONES} \
    env.steps=1500 trainer.update_log_every=500 trainer.save_every=1e9 \
    logdir=/content/prueba_random > /content/prueba_random.log 2>&1; \
    tail -n 5 /content/prueba_random.log

In [ ]:
import json

pasos = [json.loads(l) for l in open("/content/prueba_random/metrics.jsonl") if '"fps/fps"' in l]
fps = [p["fps/fps"] for p in pasos if p["fps/fps"] > 0]
if not fps:
    print("No alcanzó a medir la velocidad. Revisa el final de /content/prueba_random.log (arriba).")
else:
    v = sum(fps) / len(fps)
    print(f"{v:.1f} pasos por segundo → {PASOS:,} pasos en unas {PASOS / v / 3600:.1f} horas "
          f"(con TRAIN_RATIO={TRAIN_RATIO})\n")
ultimo_video("/content/prueba_random")

## 8. Abrir TensorBoard

Ábrelo **antes** de entrenar: mientras la celda de entrenamiento corre, Colab no deja ejecutar otras, pero TensorBoard se sigue actualizando (botón de recargar arriba a la derecha).

Lo más informativo para este experimento, en la pestaña Scalars:

- `train/loss/image`: qué tan bien reconstruye lo que ve.
- `train/loss/dyn` y `train/loss/rep`: qué tan bien predice el siguiente estado.
- `train/loss/rew`: si aprende dónde está la recompensa, aunque la política al azar casi nunca llegue a ella.

Los videos de predicción (`open_loop`) están en la pestaña Images.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "/content/drive/MyDrive/galleta"

## 9. Entrenar el world model con acciones al azar

La velocidad aparece en la salida como `fps/fps` (pasos por segundo), cada 2500 pasos. Para parar antes, usa el botón de detener de la celda: el último checkpoint queda en Drive.

In [ ]:
!cd /content/r2dreamer-cookie && .venv/bin/python -u train.py {OPCIONES} \
    env.steps={PASOS} trainer.update_log_every=2500 trainer.save_every=1e4 buffer.max_size=1e5 \
    logdir="{LOGDIR}"

## 10. Ver lo que imagina el modelo

Córrelo cuando el entrenamiento termine o lo detengas. Muestra el último video de predicción guardado, con tres filas:

1. **Arriba:** lo que de verdad pasó.
2. **Al medio:** lo que predice el modelo. Los primeros 5 cuadros los ve (reconstrucción); desde ahí imagina el resto solo con las acciones, sin ver nada más.
3. **Abajo:** el error. Gris es que acertó; claro u oscuro es que se equivocó.

Cada columna es una secuencia distinta sacada del buffer.

In [ ]:
ultimo_video(LOGDIR)